# Which partners and priors drive a prediction

A PWPA prediction is a weighted average over partners, and each partner's score is a sum over priors, so every prediction splits into partners and priors.
This notebook shows the top partners of LCK in resting and stimulated Marson T cells, with each partner's score split by prior (`figures/partners_lck.png`).

It explains the final model, which is fitted once on every Marson knockdown, lockbox fold included (`marson.final_model`).
LCK's own knockdown is never its partner, but the gates have seen it: this explains the model, it is not a held-out prediction.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.append("../utils")
import marson
from paths import data_dir

sys.path.append("../1.model")
import pwpa

torch.set_num_threads(int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count())))  # marson.final_model fits on first use

In [ ]:
obs, delta, readouts, symbols = marson.load()
priors = marson.priors(symbols)  # PIE's five genetic sources, looked up by each gene's symbol
best = marson.final_model(priors)  # fitted on first use; 1.gene_regulatory_network.ipynb uses the same fit
names, gates = best["priors"], best["gates"].mean(0).numpy()
chosen = [priors[n] for n in names]
pd.Series(gates, names).round(4)

The part of prior `d` in the score of partner `j` is `gate_d × z_d(LCK, j)`, with the gate averaged over the 5 seeds.
The parts add up to the score, and the weights come from the scores of all partners, averaged over seeds as in `pwpa.predict`.

In [ ]:
GENE, N_TOP = "LCK", 5
gene = np.array([g for g, s in symbols.items() if s == GENE])
assert len(gene) == 1

panels = {}
for state in ("Rest", "Stim48hr"):
    m = (obs.condition == state).values
    partners = obs.ensg.values[m]
    w = pwpa.predict(chosen, best["scale"], best["gates"], gene, partners, delta[m])[1][0]
    top = np.argsort(-w)[:N_TOP]
    z = pwpa.standardise(*pwpa.similarities(chosen, gene, partners[top]), best["scale"]).numpy()[0]
    parts = pd.DataFrame(z * gates, index=obs.symbol.values[m][top], columns=names)
    panels[state] = parts.assign(weight=w[top]), int((w > 0).sum())
    print(f"{state}: {(w > 0).sum()} partners with weight, {w[top].sum():.0%} on the top {N_TOP}")
    display(panels[state][0].round(3))

In [ ]:
STYLE = {  # label and colour per prior
    "perturbation_text": ("Perturbation description", "#6547b0"),
    "ncbi_text": ("NCBI gene annotations", "#9b86d6"),
    "esm2": ("ESM2", "#c9822f"),
    "string_space": ("STRING", "#8d97a8"),
    "depmap_gene_effect": ("DepMap", "#2c6aa8"),
}
TITLE = {"Rest": "Rest", "Stim48hr": "Stim 48 h"}
shown = [n for n in names if any((p[n].abs() > 1e-3).any() for p, _ in panels.values())]

fig, axes = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)
for ax, (state, (parts, nz)) in zip(axes, panels.items()):
    y = -np.arange(len(parts))
    pos, neg = np.zeros(len(parts)), np.zeros(len(parts))
    for n in shown:  # parts above 0 stack to the right of 0, parts below 0 to the left
        v = parts[n].values
        label, colour = STYLE.get(n, (n, None))
        ax.barh(y, v, left=np.where(v > 0, pos, neg), height=0.65, color=colour, label=label)
        pos, neg = pos + v.clip(min=0), neg + v.clip(max=0)
    ax.axvline(0, color="k", lw=1.2)
    ax.set_yticks(y, parts.index, family="monospace")
    for yi, w in zip(y, parts.weight):
        ax.annotate(f"{w:.1%}", (1, yi), xycoords=("axes fraction", "data"), xytext=(8, 0), textcoords="offset points", va="center")
    ax.set_title(f"{TITLE[state]}: top {N_TOP} of {nz} partners with weight", loc="left", fontweight="bold")
    ax.spines[["top", "right", "left"]].set_visible(False)
axes[0].annotate("Weight", (1, 1), xycoords="axes fraction", xytext=(8, 4), textcoords="offset points", color="0.4")
axes[1].set_xlabel("← lowers the score    part of the partner's score    raises the score →")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(0.5, 0), ncol=3, frameon=False)
fig.suptitle(f"Top {N_TOP} partners of {GENE}: score by prior", x=0.02, ha="left", fontweight="bold")
fig.tight_layout()
fig.savefig("figures/partners_lck.png", dpi=200, bbox_inches="tight")